# Flash Flash Revolution Silver Layer VerticalDensity Calculation

This notebook calculates **VerticalDensity**, a metric measuring the intensity of note patterns in Flash Flash Revolution songs. VerticalDensity is defined as the reciprocal of the time delta between consecutive notes of the same orientation, representing how rapidly a player must hit the same arrow.

**Key Features:**
- **Note Decomposition**: Multi-orientation notes (e.g., "1001" for left+right arrows) are decomposed into separate single-orientation rows to accurately reflect per-finger intensity
- **Incremental Processing**: Uses `swf_version` tracking to detect new/updated songs and only processes changed data
- **Note-Level Data**: Stores VerticalDensity for each note (not aggregated)
- **First Note Handling**: The first note of each song+orientation has vertical_density = 0 (no previous note to compare)

**Source Table:**
- `acubed.ffr.silver__notes-adjusted`: Note-level data with timestamps and orientations

**Target Table:**
- `acubed.ffr.`silver__vertical-density``: Note-level VerticalDensity data with swf_version for change tracking

**Schema:**
- `song_id` (bigint): Unique song identifier
- `note_id` (int): Note sequence number
- `timestamp` (double): Note timestamp
- `orientation` (string): Single orientation after decomposition (e.g., "1000", "0100", "0010", "0001")
- `prev_timestamp` (double): Timestamp of previous note with same orientation (NULL for first note)
- `time_delta` (double): Time difference between consecutive notes (NULL for first note)
- `vertical_density` (double): Reciprocal of time_delta (notes per second); 0 for first note of each orientation
- `swf_version` (long): Version tracking for incremental updates

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lag, explode, array, when, lit, count, 
    mean, max as spark_max, min as spark_min, stddev, expr
)
from delta.tables import DeltaTable

print("✓ Imports loaded successfully")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.`silver__vertical-density`"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

silver_table = "acubed.ffr.`silver__vertical-density`"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs and updated songs
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_vertical_density = spark.table(silver_table).select(
        "song_id",
        "swf_version"
    )
    
    # Find new songs (in bronze but not in silver)
    new_songs = bronze_songlist.join(
        silver_vertical_density,
        "song_id",
        "left_anti"
    ).select("song_id", "bronze_swf_version")
    
    # Find updated songs (swf_version changed)
    updated_songs = bronze_songlist.alias("bronze").join(
        silver_vertical_density.alias("silver"),
        col("bronze.song_id") == col("silver.song_id"),
        "inner"
    ).filter(
        col("bronze.bronze_swf_version") != col("silver.swf_version")
    ).select(
        col("bronze.song_id"),
        col("bronze.bronze_swf_version")
    )
    
    # Combine new and updated songs
    all_changed = new_songs.union(updated_songs)
    changed_count = all_changed.count()
    
    if changed_count == 0:
        print("ℹ No changed songs detected")
        changed_song_ids = None
    else:
        changed_song_ids = all_changed.select("song_id").distinct()
        print(f"✓ Found {changed_count} songs to process (new or updated)")
else:
    if process_all:
        print("ℹ Force full refresh mode enabled (process_all=True)")
    else:
        print("ℹ First run - no existing silver table found")
    print("✓ Will process all songs")
    changed_song_ids = None  # Set to None for full refresh

In [0]:
# Determine processing mode based on change detection
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping VerticalDensity calculation")
    print("✓ Notebook will complete successfully (idempotent run)")
    # Set empty DataFrame to allow downstream cells to run
    df_density_with_version = spark.createDataFrame([], schema="song_id long, note_id int, timestamp double, orientation string, prev_timestamp double, time_delta double, vertical_density double, swf_version long")
else:
    # Load silver notes adjusted table
    df_notes = spark.table("acubed.ffr.`silver__notes-adjusted`")
    
    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_notes = df_notes.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Processing {df_notes.select('song_id').distinct().count()} changed songs")
    else:
        print(f"ℹ Full refresh mode: Processing all {df_notes.select('song_id').distinct().count()} songs")
    
    print("\nStep 1: Decomposing multi-orientation notes into single orientations...")
    print("=" * 60)
    
    # Decompose multi-orientation notes into separate rows
    # For example: "1001" becomes two rows: "1000" and "0001"
    df_decomposed = df_notes.select(
        "song_id",
        "note_id",
        "timestamp",
        "orientation"
    ).withColumn(
        # Create an array of active orientations based on the bit pattern
        "single_orientations",
        array(
            when(col("orientation").substr(1, 1) == "1", lit("1000")).otherwise(lit(None)),
            when(col("orientation").substr(2, 1) == "1", lit("0100")).otherwise(lit(None)),
            when(col("orientation").substr(3, 1) == "1", lit("0010")).otherwise(lit(None)),
            when(col("orientation").substr(4, 1) == "1", lit("0001")).otherwise(lit(None))
        )
    ).select(
        "song_id",
        "note_id",
        "timestamp",
        explode("single_orientations").alias("orientation")
    ).filter(
        col("orientation").isNotNull()
    )
    
    print(f"✓ Decomposed notes into single orientations")
    
    print("\nStep 2: Calculating VerticalDensity...")
    print("=" * 60)
    
    # Define window partitioned by song and orientation, ordered by timestamp
    window_spec = Window.partitionBy("song_id", "orientation").orderBy("timestamp", "note_id")
    
    # Calculate time delta and vertical density
    df_density = df_decomposed.withColumn(
        "prev_timestamp",
        lag("timestamp").over(window_spec)
    ).withColumn(
        "time_delta",
        when(col("prev_timestamp").isNotNull(), col("timestamp") - col("prev_timestamp")).otherwise(lit(None))
    ).withColumn(
        "vertical_density",
        when(
            col("time_delta").isNotNull() & (col("time_delta") > 0),
            1.0 / col("time_delta")
        ).otherwise(lit(0.0))  # First note of each orientation gets 0
    )
    
    print(f"✓ Calculated vertical_density for {df_density.count()} notes")
    
    print("\nStep 3: Adding swf_version for change tracking...")
    print("=" * 60)
    
    # Join with bronze_songlist to add swf_version
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version")
    )
    
    df_density_with_version = df_density.join(
        bronze_songlist,
        "song_id",
        "inner"
    ).select(
        "song_id",
        "note_id",
        "timestamp",
        "orientation",
        "prev_timestamp",
        "time_delta",
        "vertical_density",
        "swf_version"
    )
    
    print(f"✓ Added swf_version to {df_density_with_version.count()} notes")
    print("\nSample output:")
    display(df_density_with_version.orderBy("song_id", "orientation", "note_id").limit(10))

In [0]:
# Save to Delta table using DELETE + INSERT pattern for incremental updates
table_name = "acubed.ffr.`silver__vertical-density`"

# Initialize variable for tracking deletions
song_ids_to_delete = []


# Capture counts BEFORE operations
rows_to_insert = df_density_with_version.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update")
    print("✓ Table remains unchanged (idempotent run)")
else:
    if spark.catalog.tableExists(table_name):
        # Table exists - use DELETE + INSERT for incremental updates
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_density_with_version.select("song_id").distinct()
        
        # Delete existing rows for these songs
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_density_with_version.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {rows_to_insert} new rows")
    else:
        # First run - create table
        df_density_with_version.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        print(f"✓ Created table {table_name} with {rows_to_insert} rows")
    
    print(f"\n✓ Updated {len(song_ids_to_delete) if spark.catalog.tableExists(table_name) and len(song_ids_to_delete) > 0 else 'all'} songs in {table_name}")

print("\n" + "=" * 60)
print("VerticalDensity calculation complete!")
print("=" * 60)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql.functions import col, mean

# Load the full vertical_density table (note-level data)
df_vertical_density = spark.table("acubed.ffr.`silver__vertical-density`")

# Aggregate to song level for visualization
df_song_avg = df_vertical_density.groupBy("song_id").agg(
    mean("vertical_density").alias("avg_density")
)

# Load song metadata
df_songlist = spark.table("acubed.ffr.bronze__songlist")

# Join with song metadata to get difficulty
df_density_difficulty = df_song_avg.join(
    df_songlist.select("id", "difficulty", "name"),
    df_song_avg.song_id == df_songlist.id,
    "inner"
).select(
    col("song_id"),
    col("name"),
    col("avg_density"),
    col("difficulty")
)

print("=" * 70)
print("VERTICALDENSITY vs DIFFICULTY VISUALIZATION")
print("=" * 70)

# Check for nulls and basic stats
total_songs = df_density_difficulty.count()
songs_with_difficulty = df_density_difficulty.filter(col("difficulty").isNotNull()).count()

print(f"\nTotal songs analyzed: {total_songs:,}")
print(f"Songs with difficulty data: {songs_with_difficulty:,}")

if songs_with_difficulty > 0:
    # Convert to pandas for plotting
    df_plot = df_density_difficulty.filter(col("difficulty").isNotNull()).toPandas()
    
    print(f"\nDifficulty range: {df_plot['difficulty'].min()} to {df_plot['difficulty'].max()}")
    print(f"Avg VerticalDensity range: {df_plot['avg_density'].min():.2f} to {df_plot['avg_density'].max():.2f}")
    
    # Create scatterplot
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Color by difficulty for better visualization
    scatter = ax.scatter(
        df_plot['avg_density'], 
        df_plot['difficulty'],
        alpha=0.6,
        s=50,
        c=df_plot['difficulty'],
        cmap='RdYlGn_r',  # Red (hard) to Green (easy)
        edgecolors='black',
        linewidth=0.5
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Difficulty', fontsize=12)
    
    # Set axis limits
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 150)
    
    # Labels and title
    ax.set_xlabel('Average VerticalDensity (notes per second)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Difficulty', fontsize=12, fontweight='bold')
    ax.set_title('Song Difficulty vs Average VerticalDensity\n(After Note Decomposition)', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Calculate and display correlation for visible range
    df_visible = df_plot[(df_plot['avg_density'] <= 10) & (df_plot['difficulty'] <= 150)]
    correlation = df_visible['avg_density'].corr(df_visible['difficulty'])
    ax.text(0.02, 0.98, f'Correlation: {correlation:.3f}\n(range: 0-10, 0-150)', 
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualization complete!")
    print(f"   Correlation: {correlation:.3f}")
else:
    print("\n⚠️ No songs with difficulty data found!")